# Final Technical Roadmap: "The Gold Standard" Strategy

## Overview
A roadmap for breaking the performance ceiling and revamping the architecture for the final production ready teacher model that will generate the suite of classification models in the final release.

---

## 0. Phase 0: Task-Adaptive Pre-Training (TAPT)
**Goal:** Domain Adaptation. Align the model's internal embeddings and tokenizer with the specific vocabulary and structure of political news reporting (e.g., "fairness" in a legislative context vs. general usage).

* **Base Model:** `allenai/longformer-base-4096`
* **Architecture:** `LongformerForMaskedLM`
* **Dataset:** Raw text of 400k+ collected articles (Unlabeled).
* **Objective:** Masked Language Modeling (MLM).
* **Technical Specifications:**
    * **Masking Rate:** 15% dynamic masking.
    * **Epochs:** 10.
    * **Sequence Length:** 2048 (or 4096 if VRAM permits).
    * **Output:** `longformer-news-base` (Saved Weights).

Following recommendations from this guide.

## 1. Phase 1: Silver Fine-Tuning (Representation Learning)
**Goal:** Feature Extraction. Teach the model general correlations between text patterns and frame labels using a massive, albeit noisy, dataset.

* **Initialization:** Weights from `longformer-news-base`.
* **Architecture:** `LongformerForSequenceClassification` (15 classes).
* **Dataset:** 400k Machine-Labeled ("Silver") Articles.
* **Objective:** Multi-Label Classification.
* **Technical Specifications:**
    * **Loss Function:** **Focal Loss** (replacing Weighted BCE).
        * *Gamma (`$\gamma$`):* 2.0 (standard starting point).
        * *Alpha (`$\alpha$`):* Balanced by class frequency.
    * **Learning Rate:** Standard (`2e-5` to `3e-5`).
    * **Global Attention:** Applied to `[CLS]` and `[TOPIC]` tokens.
    * **Output:** `longformer-news-silver`.

## 2. Phase 2: Gold Fine-Tuning (Boundary Refinement)
**Goal:** Precision Correction. Realign decision boundaries to match human nuance, effectively treating the Silver phase as a "warm-up" and the Gold phase as the "truth."

* **Initialization:** Weights from `longformer-news-silver`.
* **Dataset:** Human-Labeled ("Gold") Data (Media Frames Corpus + SemEval).
* **Technical Specifications:**
    * **Learning Rate:** **Strictly Low** (`1e-6` to `5e-6`) to prevent catastrophic forgetting.
    * **Loss Function:** Focal Loss (continues from Phase 1).
    * **Regularization:** High weight decay or Early Stopping to prevent overfitting on the smaller dataset.
    * **Output:** `longformer-news-gold` (Production Model).

---

## Key Architectural Decisions

### A. Architecture: Longformer-Base-4096
* **Decision:** Retain Longformer over RoBERTa/Muppet.
* **Justification:** The "Context Switching" error analysis revealed that defining frames often requires linking disparate parts of a document (e.g., title vs. conclusion). Longformer's global attention mechanism is the only efficient way to capture these long-range dependencies without aggressive truncation loss.

### B. Loss Function: Focal Loss
* **Decision:** Replace Static Class Weights with Focal Loss.
* **Justification:**
    * *Problem:* In multi-label framing, "easy negatives" (e.g., knowing a Sports article isn't about Nuclear Safety) dominate the loss calculation, washing out the gradient signal from difficult, nuanced examples.
    * *Solution:* Focal Loss dynamically down-weights easy examples, forcing the model to focus 100% of its capacity on the "hard" examples where it is confused (e.g., distinguishing "Economic" from "Capacity & Resources").